In [1]:
import pandas as pd
import os

In [2]:
Uniprot_mapping = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/data/raw_data/DRBP_uniprot_mapping.csv')
Uniprot_mapping.drop_duplicates(subset='ENSP_ID',inplace=True)
Uniprot_mapping

,ENSP_ID,Uniprot,Sequence,Class,Source
0,>ENSP00000345702,P23511,MEQYTANSNSSTEQIVVQAGQIQQQQQGGVTAVQLQTEAQVASASG...,1,DBP
2,>ENSP00000317128,Q9Y4D7,MAPRAAGGAPLSARAAAASPPPFQTPPRCPVPLLLLLLLGAARAGA...,0,NDBP
4,>ENSP00000383042,O60341,MLSGKKAAAAAAAAAAAATGTEAGPGTAGGSENGSEVAAQPAGLSG...,1,DBP
6,>ENSP00000368332,Q96QS3,MSNQYQEEGCSERPECKSKSPTLLSSYCIDSILGRRSPCKMRLLGA...,0,NDBP
8,>ENSP00000006015,P31270,MDFDERGPCSSNMYLPSCTYYVSGPDFSSLPSFLPQTPSSRPMTYS...,0,NDBP
...,...,...,...,...,...
36428,>ENSP00000310632,Q8NH09,MALGNHSTITEFLLLGLSADPNIRALLFVLFLGIYLLTIMENLMLL...,0,NDBP
36430,>ENSP00000493167,P0DPE3,MAARTLASALVLTLWVWALAPAGAVDAMGPHAAVRLAELLTPEECG...,0,NDBP
36432,>ENSP00000314295,P0DPD7,MASPGAGRAPPELPERNCGYREVEYWDQRYQGAADSAPYDWFGDFS...,0,NDBP
36434,>ENSP00000384223,P0DPD8,MASPGAGRAPPELPERNCGYREVEYWDQRYQGAADSAPYDWFGDFS...,0,NDBP


In [4]:
human_interpro = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/analysis_raw_data/interproscan/Interproscan_result_250704.tsv',sep='\t',header=None)
human_interpro.columns=['ENSP_ID','ID','Length','Source_DB','Source_ID','Description','Start','End','p-value','t','date','Interpro_ID','Description2','GO','etc']
human_interpro = human_interpro[human_interpro['Interpro_ID'] != '-']
human_interpro

,ENSP_ID,ID,Length,Source_DB,Source_ID,Description,Start,End,p-value,t,date,Interpro_ID,Description2,GO,etc
0,ENSP00000352785,bf8fed9e55b18f0adb4b5b7cb2404678,1059,PANTHER,PTHR24025,DESMOGLEIN FAMILY MEMBER,43,688,1.0E-176,T,05-07-2025,IPR050971,Cadherin domain-containing protein,GO:0005509(PANTHER)|GO:0005911(PANTHER)|GO:003...,-
1,ENSP00000352785,bf8fed9e55b18f0adb4b5b7cb2404678,1059,Pfam,PF01049,"Cadherin, Y-type LIR-motif",810,866,0.0081,T,05-07-2025,IPR000233,"Cadherin, Y-type LIR-motif",GO:0005509(InterPro)|GO:0007156(InterPro),-
2,ENSP00000352785,bf8fed9e55b18f0adb4b5b7cb2404678,1059,ProSiteProfiles,PS50268,Cadherins domain profile.,76,157,18.617403,T,05-07-2025,IPR002126,Cadherin-like,GO:0005509(InterPro)|GO:0007156(InterPro)|GO:0...,-
3,ENSP00000352785,bf8fed9e55b18f0adb4b5b7cb2404678,1059,SUPERFAMILY,SSF49313,Cadherin-like,75,157,9.85E-13,T,05-07-2025,IPR015919,Cadherin-like superfamily,GO:0005509(InterPro)|GO:0016020(InterPro),-
4,ENSP00000352785,bf8fed9e55b18f0adb4b5b7cb2404678,1059,ProSiteProfiles,PS50268,Cadherins domain profile.,386,497,17.345886,T,05-07-2025,IPR002126,Cadherin-like,GO:0005509(InterPro)|GO:0007156(InterPro)|GO:0...,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
387980,ENSP00000219611,37766b056a6e3240090d3fd76576c001,1086,SMART,SM00547,zf_4,46,70,4.0E-5,T,05-07-2025,IPR001876,"Zinc finger, RanBP2-type",-,-
387981,ENSP00000219611,37766b056a6e3240090d3fd76576c001,1086,SMART,SM00547,zf_4,342,366,0.048,T,05-07-2025,IPR001876,"Zinc finger, RanBP2-type",-,-
387982,ENSP00000219611,37766b056a6e3240090d3fd76576c001,1086,SMART,SM00547,zf_4,5,29,2.1E-7,T,05-07-2025,IPR001876,"Zinc finger, RanBP2-type",-,-
387983,ENSP00000219611,37766b056a6e3240090d3fd76576c001,1086,SMART,SM00547,zf_4,414,438,8.8E-7,T,05-07-2025,IPR001876,"Zinc finger, RanBP2-type",-,-


In [5]:
import pandas as pd
from collections import Counter
import numpy as np

def split_into_occurrences(intervals, join_gap=0):
    """
    intervals: list of (start, end)
    반환: [ [ (s,e), (s,e), ... ],  [ ... ], ... ]  # 겹치는 것끼리 묶은 클러스터들
    """
    ints = sorted([(int(s), int(e)) if s <= e else (int(e), int(s)) for s, e in intervals])
    clusters = []
    cur = []
    cur_end = None

    for s, e in ints:
        if not cur:
            cur = [(s, e)]
            cur_end = e
        else:
            # join_gap 허용: 다음 구간의 시작이 현재 끝+join_gap 이하면 같은 occurrence로 묶음
            if s <= cur_end + join_gap:
                cur.append((s, e))
                cur_end = max(cur_end, e)
            else:
                clusters.append(cur)
                cur = [(s, e)]
                cur_end = e
    if cur:
        clusters.append(cur)
    return clusters

def majority_voting_consensus(intervals, choose="longest"):
    """
    intervals: [(start, end), ...]
    choose:
      - "longest": 최대 커버리지(빈도) 연속구간들 중 가장 긴 구간
      - "span":    최대 커버리지 위치들의 전체 min~max
      - "median":  입력 구간들의 중앙에 가장 가까운 최대 커버리지 연속구간
    """
    pos_counts = Counter()
    for s, e in intervals:
        for pos in range(s, e+1):
            pos_counts[pos] += 1
    if not pos_counts:
        return None, None

    max_count = max(pos_counts.values())
    max_pos = sorted([p for p, c in pos_counts.items() if c == max_count])

    if choose == "span":
        return max_pos[0], max_pos[-1]

    # 연속 구간 분해
    segs = []
    start = prev = None
    for p in max_pos:
        if start is None:
            start = prev = p
        elif p == prev + 1:
            prev = p
        else:
            segs.append((start, prev))
            start = prev = p
    if start is not None:
        segs.append((start, prev))

    if choose == "longest":
        return max(segs, key=lambda x: (x[1]-x[0], -x[0]))

    if choose == "median":
        mids = [(s+e)/2 for s, e in intervals]
        center = np.median(mids) if mids else (segs[0][0] + segs[0][1]) / 2
        return min(segs, key=lambda x: abs((x[0]+x[1])/2 - center))

    return max_pos[0], max_pos[-1]

def get_occurrence_level_consensus(df, join_gap=0, choose="longest", cap_end_at=982):
    """
    df: columns = ['ENSP_ID','Interpro_ID','Start','End']
    반환: occurrence 단위의 consensus 좌표
    """
    out_rows = []
    for (ens, ipr), g in df.groupby(['ENSP_ID', 'Interpro_ID']):
        # 동일 IPR의 모든 구간을 occurrence 별로 분리
        intervals = [(int(s), int(e)) for s, e in zip(g['Start'], g['End']) if pd.notna(s) and pd.notna(e)]
        clusters = split_into_occurrences(intervals, join_gap=join_gap)

        # 각 occurrence(cluster)마다 consensus 계산
        for occ_idx, cluster in enumerate(clusters, start=1):
            cs, ce = majority_voting_consensus(cluster, choose=choose)
            if cs is None:
                continue
            if cap_end_at is not None and ce is not None:
                ce = min(int(ce), cap_end_at)
            if cs < (cap_end_at if cap_end_at is not None else float('inf')):
                out_rows.append({
                    'ENSP_ID': ens,
                    'Interpro_ID': ipr,
                    'Occurrence': occ_idx,
                    'consensus_start': int(cs),
                    'consensus_end': int(ce)
                })
    return pd.DataFrame(out_rows)

In [6]:
# dna_interpro: ['ENSP_ID','Interpro_ID','Start','End'] 포함
consensus_df_span = get_occurrence_level_consensus(
    human_interpro,
    join_gap=0,        # 간격 허용치(필요하면 5~10 같은 값으로 조정)
    choose="span",  # 최대 커버리지의 가장 긴 연속 구간
    cap_end_at=1022     # 좌표 상한선이 필요 없으면 None
)
consensus_df_span

,ENSP_ID,Interpro_ID,Occurrence,consensus_start,consensus_end
0,ENSP00000000233,IPR005225,1,16,138
1,ENSP00000000233,IPR006689,1,19,140
2,ENSP00000000233,IPR024156,1,1,174
3,ENSP00000000233,IPR027417,1,14,177
4,ENSP00000000233,IPR045872,1,18,176
...,...,...,...,...,...
118674,ENSP00000498205,IPR012340,3,534,615
118675,ENSP00000498205,IPR012340,4,631,710
118676,ENSP00000498205,IPR012340,5,719,801
118677,ENSP00000498205,IPR045209,1,1,1022


In [13]:
consensus_df_span["ENSP_ID"].value_counts().mean()

6.529793672627235

In [14]:
consensus_df_span["ENSP_ID"].value_counts().median()

5.0

In [7]:
consensus_df_span.drop_duplicates(subset='ENSP_ID')

,ENSP_ID,Interpro_ID,Occurrence,consensus_start,consensus_end
0,ENSP00000000233,IPR005225,1,16,138
5,ENSP00000000412,IPR000296,1,19,42
17,ENSP00000000442,IPR000536,1,230,392
28,ENSP00000001008,IPR001179,1,50,134
38,ENSP00000001146,IPR001128,1,441,441
...,...,...,...,...,...
118639,ENSP00000498164,IPR003307,1,649,724
118646,ENSP00000498177,IPR005756,1,9,122
118652,ENSP00000498186,IPR000111,1,126,142
118659,ENSP00000498190,IPR001356,1,109,165


In [9]:
consensus_df_span['seq_len'] = consensus_df_span['consensus_end'] - consensus_df_span['consensus_start'] + 1
consensus_df_span['seq_len'].mean()

136.10648893233008

In [11]:

# 2) Interpro_ID 전체 빈도 계산 (각 단백질에서 한 번만 카운트됨)
ipr_counts = consensus_df_span["Interpro_ID"].value_counts()

ipr_counts

Interpro_ID
IPR013087    7202
IPR013783    2117
IPR000276    1899
IPR036236    1385
IPR003591    1337
             ... 
IPR030565       1
IPR040329       1
IPR050793       1
IPR038752       1
IPR048059       1
Name: count, Length: 17606, dtype: int64

In [13]:
consensus_df_span.to_csv('/Users/hanjin/Desktop/Project_ongoing/DRBP/data/DRBP_result_analysis/interpro_consensus/human_interpro_consensus_span.csv', index=False)

In [10]:
# dna_interpro: ['ENSP_ID','Interpro_ID','Start','End'] 포함
consensus_df_longest = get_occurrence_level_consensus(
    human_interpro,
    join_gap=0,        # 간격 허용치(필요하면 5~10 같은 값으로 조정)
    choose="longest",  # 최대 커버리지의 가장 긴 연속 구간
    cap_end_at=982     # 좌표 상한선이 필요 없으면 None
)
consensus_df_longest

,ENSP_ID,Interpro_ID,Occurrence,consensus_start,consensus_end
0,ENSP00000000233,IPR005225,1,16,138
1,ENSP00000000233,IPR006689,1,74,99
2,ENSP00000000233,IPR024156,1,1,174
3,ENSP00000000233,IPR027417,1,14,177
4,ENSP00000000233,IPR045872,1,18,176
...,...,...,...,...,...
117998,ENSP00000498205,IPR012340,3,534,615
117999,ENSP00000498205,IPR012340,4,631,710
118000,ENSP00000498205,IPR012340,5,719,801
118001,ENSP00000498205,IPR045209,1,1,982


In [9]:
def map_to_A_column(ensp_id: str, A_columns: set):
    """B의 ENSP_ID를 A의 컬럼명으로 매핑(> 유무 보정)"""
    if ensp_id in A_columns:
        return ensp_id
    if f">{ensp_id}" in A_columns:
        return f">{ensp_id}"
    if ensp_id.startswith('>') and ensp_id[1:] in A_columns:
        return ensp_id[1:]
    return np.nan  # 매칭 실패 시 NaN

def attach_means(A: pd.DataFrame, B: pd.DataFrame) -> pd.DataFrame:
    B = B.copy()

    # 1) ENSP_ID -> A 컬럼명 매핑
    A_cols = set(A.columns)
    B['A_col'] = B['ENSP_ID'].astype(str).map(lambda x: map_to_A_column(x, A_cols))

    # 2) 컬럼 전체 평균 (col-wise mean)
    col_means = A.mean(axis=0, skipna=True)
    B['mean_score_all_sequence'] = B['A_col'].map(col_means)

    # 3) 구간 평균 (start~end 포함)
    n_rows = len(A)

    def range_mean(row):
        col = row['A_col']
        if pd.isna(col) or col not in A.columns:
            return np.nan
        # 숫자화 & 정리
        try:
            s = int(row['consensus_start'])
            e = int(row['consensus_end'])
        except Exception:
            return np.nan
        if s > e:
            s, e = e, s
        # 인덱스 클리핑 (inclusive)
        s = max(0, s)
        e = min(n_rows - 1, e)
        if s > e:
            return np.nan
        return float(A[col].iloc[s:e+1].mean())

    B['mean_score_domain_range'] = B.apply(range_mean, axis=1)

    return B

In [ ]:
dbp = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/interpret/DRBP_interpret/VATP_final/DBP_VAT_score.csv')
ndbp = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/interpret/DRBP_interpret/VATP_final/NDBP_VAT_score.csv')

DBP_concat = pd.concat([dbp,ndbp],axis=1)
DBP_concat

,>ENSP00000366221,>ENSP00000387699,>ENSP00000237853,>ENSP00000222728,>ENSP00000363193,>ENSP00000453793,>ENSP00000306335,>ENSP00000290524,>ENSP00000356857,>ENSP00000381840,...,>ENSP00000246186,>ENSP00000343709,>ENSP00000278742,>ENSP00000255381,>ENSP00000358582,>ENSP00000365730,>ENSP00000233948,>ENSP00000263119,>ENSP00000355866,>ENSP00000401633
0,0.749158,0.068893,0.377819,0.248894,0.161736,0.365515,0.248365,0.376047,0.540260,1.000000,...,0.437903,0.315410,0.357920,0.101943,0.443869,0.197823,0.082549,0.288975,0.162421,0.187645
1,0.607752,0.056657,0.339271,0.113202,0.128355,0.301721,0.204486,0.243051,0.305585,0.641289,...,0.389125,0.344811,0.250726,0.088383,0.392081,0.177802,0.080510,0.277773,0.172160,0.141917
2,0.554441,0.055133,0.333000,0.097009,0.116543,0.292882,0.215510,0.267967,0.208359,0.606201,...,0.486438,0.218981,0.282730,0.088719,0.399763,0.153710,0.075703,0.256667,0.157301,0.155810
3,0.492151,0.062338,0.307002,0.129541,0.176044,0.402027,0.161855,0.239692,0.273065,0.539936,...,0.271046,0.128385,0.274454,0.080962,0.392943,0.151249,0.073151,0.281715,0.146934,0.178445
4,0.437418,0.062445,0.307240,0.134896,0.144908,0.436072,0.238674,0.213362,0.230641,0.909994,...,0.432538,0.139753,0.306331,0.082121,0.456118,0.158346,0.075815,0.215599,0.197202,0.162223
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
977,0.255665,0.000000,0.000000,0.000000,0.074115,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.081670,0.000000,0.000000,0.000000,0.125522,0.000000,0.117690
978,0.323405,0.000000,0.000000,0.000000,0.084940,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.099861,0.000000,0.000000,0.000000,0.101315,0.000000,0.141444
979,0.285330,0.000000,0.000000,0.000000,0.062401,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.073852,0.000000,0.000000,0.000000,0.108563,0.000000,0.121639
980,0.385305,0.000000,0.000000,0.000000,0.063369,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.080777,0.000000,0.000000,0.000000,0.112646,0.000000,0.143578


In [ ]:
rbp = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/interpret/DRBP_interpret/VATP_final/RBP_VAT_score.csv')
nrbp = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/interpret/DRBP_interpret/VATP_final/NRBP_VAT_score.csv')

RBP_concat = pd.concat([rbp,nrbp],axis=1)
RBP_concat

,>ENSP00000345412,>ENSP00000442266,>ENSP00000394682,>ENSP00000253108,>ENSP00000330836,>ENSP00000254940,>ENSP00000401475,>ENSP00000348111,>ENSP00000355094,>ENSP00000246194,...,>ENSP00000369573,>ENSP00000302783,>ENSP00000382858,>ENSP00000307822,>ENSP00000355938,>ENSP00000413152,>ENSP00000363500,>ENSP00000262498,>ENSP00000412130,>ENSP00000371800
0,0.475003,0.222971,0.585433,0.334766,0.194829,0.775965,0.429231,0.082181,0.091421,0.582782,...,0.104106,0.690729,0.285324,0.208263,0.104026,0.298931,0.182363,0.374259,0.436227,0.225688
1,0.389360,0.210138,0.247764,0.116385,0.117007,1.000000,0.340749,0.084814,0.109744,0.342570,...,0.132876,0.516780,0.149704,0.193059,0.084421,0.214109,0.106780,0.272930,0.422251,0.112095
2,0.486718,0.209010,0.190010,0.097042,0.114454,0.466557,0.289597,0.062822,0.079931,0.376299,...,0.079267,0.530015,0.178663,0.170591,0.097884,0.159166,0.140449,0.202994,0.494429,0.148159
3,0.444377,0.188504,0.196824,0.106722,0.107361,0.511691,0.248385,0.072788,0.074774,0.233084,...,0.086172,0.442546,0.166491,0.174726,0.105302,0.159083,0.103213,0.216238,0.320769,0.130332
4,0.488047,0.289761,0.259700,0.100080,0.105263,0.287463,0.279624,0.077769,0.072807,0.221557,...,0.085765,0.593002,0.141914,0.184486,0.090948,0.267633,0.116068,0.269822,0.342849,0.152953
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
977,0.000000,0.000000,0.141981,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
978,0.000000,0.000000,0.161907,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
979,0.000000,0.000000,0.153261,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
980,0.000000,0.000000,0.158291,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [ ]:
DBP_concat_sorted = DBP_concat[sorted(DBP_concat.columns)]
DBP_concat_sorted

,>ENSP00000000233,>ENSP00000000412,>ENSP00000000442,>ENSP00000001008,>ENSP00000001146,>ENSP00000002125,>ENSP00000002165,>ENSP00000002596,>ENSP00000002829,>ENSP00000003084,...,>ENSP00000498104,>ENSP00000498110,>ENSP00000498115,>ENSP00000498157,>ENSP00000498161,>ENSP00000498164,>ENSP00000498177,>ENSP00000498186,>ENSP00000498190,>ENSP00000498205
0,0.352806,0.187579,0.115798,0.186696,0.423667,0.154984,0.098882,0.165046,0.409518,0.172262,...,0.695205,0.970111,0.119977,0.130625,0.357986,0.235200,0.721864,0.251281,0.327899,0.307689
1,0.230668,0.165804,0.081773,0.144707,0.393610,0.126529,0.131686,0.120028,0.346989,0.156934,...,0.274429,1.000000,0.110341,0.100669,0.233561,0.192356,0.395563,0.267607,0.152558,0.165571
2,0.248216,0.145666,0.076536,0.129737,0.449467,0.137597,0.092677,0.117722,0.344584,0.171273,...,0.369662,0.498847,0.123389,0.120614,0.216177,0.179904,0.453333,0.190885,0.177403,0.192444
3,0.153386,0.154548,0.077838,0.131141,0.421580,0.133702,0.104038,0.166294,0.398856,0.163306,...,0.249224,0.690014,0.124120,0.096190,0.231296,0.200510,0.433156,0.275248,0.135254,0.222106
4,0.224998,0.128218,0.110102,0.121090,0.414706,0.183700,0.095785,0.171232,0.330890,0.141550,...,0.269149,0.525292,0.131923,0.112586,0.244600,0.183736,0.407489,0.247566,0.190468,0.221911
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
977,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.075950,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.181638
978,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.115264,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.131767
979,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.100544,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.334562
980,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.090567,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.144026


In [ ]:
RBP_concat_sorted = RBP_concat[sorted(RBP_concat.columns)]
RBP_concat_sorted

,>ENSP00000000233,>ENSP00000000412,>ENSP00000000442,>ENSP00000001008,>ENSP00000001146,>ENSP00000002125,>ENSP00000002165,>ENSP00000002596,>ENSP00000002829,>ENSP00000003084,...,>ENSP00000498104,>ENSP00000498110,>ENSP00000498115,>ENSP00000498157,>ENSP00000498161,>ENSP00000498164,>ENSP00000498177,>ENSP00000498186,>ENSP00000498190,>ENSP00000498205
0,0.381738,0.215430,0.088177,0.206652,0.486500,0.230237,0.151497,0.228763,0.344080,0.169985,...,0.609022,0.875408,0.141474,0.153629,0.328098,0.176830,0.702151,0.252063,0.328910,0.419767
1,0.295941,0.197670,0.059921,0.175564,0.423019,0.162353,0.154872,0.136763,0.286640,0.163083,...,0.260116,1.000000,0.130867,0.131750,0.270489,0.129464,0.387371,0.278982,0.146431,0.225681
2,0.297256,0.179960,0.062146,0.165813,0.424772,0.143027,0.119227,0.130260,0.295328,0.167008,...,0.361239,0.568101,0.135905,0.128911,0.256871,0.124643,0.434272,0.205046,0.165458,0.199042
3,0.218822,0.194394,0.060968,0.144398,0.422783,0.150965,0.136463,0.198288,0.344281,0.167202,...,0.248366,0.587214,0.138438,0.144426,0.311773,0.145420,0.450945,0.284451,0.133206,0.229975
4,0.267035,0.170182,0.075150,0.138624,0.423352,0.157222,0.139581,0.247282,0.294937,0.143202,...,0.247345,0.463095,0.144364,0.120122,0.289864,0.158415,0.429697,0.265839,0.194836,0.218703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
977,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.064956,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.239819
978,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.102474,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.164729
979,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.082284,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.437028
980,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.073576,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.172224


In [ ]:
# 사용 예시
DBP_with_means = attach_means(DBP_concat_sorted, consensus_df)
DBP_with_means

,ENSP_ID,Interpro_ID,Occurrence,consensus_start,consensus_end,A_col,mean_score_all_sequence,mean_score_domain_range
0,ENSP00000000233,IPR005225,1,16,138,>ENSP00000000233,0.059890,0.332567
1,ENSP00000000233,IPR006689,1,74,99,>ENSP00000000233,0.059890,0.356560
2,ENSP00000000233,IPR024156,1,1,174,>ENSP00000000233,0.059890,0.325632
3,ENSP00000000233,IPR027417,1,14,177,>ENSP00000000233,0.059890,0.336811
4,ENSP00000000233,IPR045872,1,18,176,>ENSP00000000233,0.059890,0.343212
...,...,...,...,...,...,...,...,...
117998,ENSP00000498205,IPR012340,3,534,615,>ENSP00000498205,0.051281,0.010122
117999,ENSP00000498205,IPR012340,4,631,710,>ENSP00000498205,0.051281,0.014302
118000,ENSP00000498205,IPR012340,5,719,801,>ENSP00000498205,0.051281,0.029561
118001,ENSP00000498205,IPR045209,1,1,982,>ENSP00000498205,0.051281,0.051019


In [ ]:
# 사용 예시
RBP_with_means = attach_means(RBP_concat_sorted, consensus_df)
RBP_with_means

,ENSP_ID,Interpro_ID,Occurrence,consensus_start,consensus_end,A_col,mean_score_all_sequence,mean_score_domain_range
0,ENSP00000000233,IPR005225,1,16,138,>ENSP00000000233,0.063139,0.344355
1,ENSP00000000233,IPR006689,1,74,99,>ENSP00000000233,0.063139,0.370706
2,ENSP00000000233,IPR024156,1,1,174,>ENSP00000000233,0.063139,0.342739
3,ENSP00000000233,IPR027417,1,14,177,>ENSP00000000233,0.063139,0.350639
4,ENSP00000000233,IPR045872,1,18,176,>ENSP00000000233,0.063139,0.356389
...,...,...,...,...,...,...,...,...
117998,ENSP00000498205,IPR012340,3,534,615,>ENSP00000498205,0.068689,0.015228
117999,ENSP00000498205,IPR012340,4,631,710,>ENSP00000498205,0.068689,0.025760
118000,ENSP00000498205,IPR012340,5,719,801,>ENSP00000498205,0.068689,0.046758
118001,ENSP00000498205,IPR045209,1,1,982,>ENSP00000498205,0.068689,0.068331


In [ ]:
DBP_with_means.to_csv('DBP_IPR_VATP_final.csv',index=None)
RBP_with_means.to_csv('RBP_IPR_VATP_final.csv',index=None)

In [6]:
ptm = pd.read_csv('/Users/khj/Desktop/Project_ongoing/DRBP/interpret/ptm_position.txt',sep='\t',header=None)
ptm[8] = ptm[6] + '-' + ptm[7]
ptm = ptm.drop_duplicates(subset=[1,2,8])
ptm

FileNotFoundError: [Errno 2] No such file or directory: '/Users/khj/Desktop/Project_ongoing/DRBP/interpret/ptm_position.txt'

In [ ]:
consensus_df = pd.merge(ptm, Uniprot_mapping, left_on=ptm[1], right_on='Uniprot', how='inner')
consensus_df = consensus_df[['ENSP_ID',8,2]]
consensus_df.columns = ['ENSP_ID','Interpro_ID','consensus_start']
consensus_df['consensus_end'] = consensus_df['consensus_start']
consensus_df = consensus_df[consensus_df['consensus_start'] < 982]

consensus_df

,ENSP_ID,Interpro_ID,consensus_start,consensus_end
0,>ENSP00000300161,S-Phosphorylation,6,6
1,>ENSP00000300161,K-Acetylation,70,70
2,>ENSP00000300161,K-Acetylation,117,117
3,>ENSP00000264335,M-Acetylation,1,1
4,>ENSP00000264335,T-Phosphorylation,38,38
...,...,...,...,...
118429,>ENSP00000333725,Y-Phosphorylation,128,128
118430,>ENSP00000438144,C-S-nitrosylation,137,137
118431,>ENSP00000384164,K-Ubiquitylation,694,694
118432,>ENSP00000431418,K-Ubiquitylation,482,482


In [ ]:
consensus_df.to_csv('/Users/khj/Desktop/Project_ongoing/DRBP/interpret/interproscan/ptm_position.csv',index=None)